# Tabu search for knapsack problem and read data, get greedy solution and verify it for TSP

## Tabu search for knapsack

### imports and global variables

In [5]:
import random, os, time

FILES_TO_TEST = [
    {"name": "../a1/knapsack-20.txt", "size": 20},
    {"name": "../a1/knapsack-200.txt", "size": 200}
]

# Different tabu tenure values to test
TABU_TENURES = [5, 10, 20]

# Max iterations budget
MAX_ITERATIONS = [100, 500, 1000]

NUM_RUNS = 30

OUTPUT_FILE = "tabu_knapsack_results.md"

### helpers

In [6]:
def load_data(file_name: str) -> tuple[list[tuple[int, int]], int]:
    """
    Loads weights, values, and max capacity from the file.
    Returns a tuple: (list_of_items, max_capacity)
    """
    items = []
    capacity = 0

    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return [], 0

    with open(file_name) as f:
        lines = f.readlines()

        for line in lines[:-1]:
            parts = line.strip().split()
            if len(parts) >= 3:
                weight = int(parts[1])
                value = int(parts[2])
                items.append((weight, value))

        try:
            capacity = int(lines[-1].strip())
        except ValueError:
            print("Error parsing capacity from last line.")

    return items, capacity


def evaluate_solution(
    solution: list[int], items: list[tuple[int, int]], max_weight: int
) -> tuple[int, int]:
    """
    Calculates total weight and value of a solution.
    Returns (weight, value).
    If weight > max_weight, value is set to 0 (penalty for infeasibility).
    """
    total_weight = 0
    total_value = 0

    for i, is_included in enumerate(solution):
        if is_included == 1:
            total_weight += items[i][0]
            total_value += items[i][1]

    # Check feasibility
    if total_weight > max_weight:
        return total_weight, 0  # Infeasible solution

    return total_weight, total_value


def get_random_solution(n: int) -> list[int]:
    """
    Generates a single random binary list of length n.
    """
    return [random.randint(0, 1) for _ in range(n)]

def N(l: list[int]) -> list[list[int]]:
    ''' 
    returns all neighbours by flipping every possible bit and returns a list of lists
    '''
    ans = []
    for i in range(len(l)):
        ans.append(l[:i] + [1-l[i]] + l[i+1:])
    return ans

def get_neighbour(l: list[int], i: int) -> list[int]:
    '''
    flips bit at locus i on a list and returns the new list
    ''' 
    return l[:i] + [1-l[i]] + l[i+1:]

### Core Algorithm

**Tabu Search** is a local search metaheuristic that uses a short-term memory structure (the *tabu list*) to escape local optima.
A move is considered *tabu* for a number of iterations equal to the *tabu tenure*, preventing revisiting recent solutions.

**Pseudocode:**
```
c = initSolution()
best = c
M = initMemory()          # tabu list: stores the iteration when each bit-flip becomes allowed again
iter = 0
while iter < max_iterations:
    x = getBestNeighbourNonTabu(c, M, iter)   # best neighbour whose move is not tabu
    updateMemory(M, move, iter, tabu_tenure)  # mark move as tabu for next `tabu_tenure` iterations
    c = x
    if value(c) > value(best): best = c
    iter += 1
return best
```

**Move representation:** a move is identified by the index of the bit being flipped.
The tabu list stores, for each bit index, the iteration number until which it is forbidden.

In [7]:
def tabu_search(
    items: list[tuple[int, int]],
    max_weight: int,
    max_iterations: int,
    tabu_tenure: int
) -> dict:
    """
    Tabu Search for the 0/1 Knapsack Problem.

    Parameters:
        items          : list of (weight, value) pairs
        max_weight     : knapsack capacity
        max_iterations : stopping criterion (number of iterations)
        tabu_tenure    : number of iterations a move stays tabu

    Returns a dict with keys: 'value', 'weight', 'solution'
    """
    n = len(items)

    # Step 1: initialise with a random solution
    current = get_random_solution(n)
    current_w, current_v = evaluate_solution(current, items, max_weight)

    best = current[:]
    best_v = current_v
    best_w = current_w

    # Step 2: initialise tabu memory
    # tabu_list[i] stores the first iteration at which flipping bit i is no longer tabu
    tabu_list = [0] * n  # all moves are allowed at iteration 0

    for iteration in range(max_iterations):
        best_neighbour = None
        best_neighbour_v = -1
        best_neighbour_w = 0
        best_move = -1

        # Step 3: evaluate all neighbours; pick the best non-tabu one
        for i in range(n):
            # Skip tabu moves unless aspiration criterion is met (see below)
            is_tabu = tabu_list[i] > iteration

            neighbour = get_neighbour(current, i)
            nw, nv = evaluate_solution(neighbour, items, max_weight)

            # Aspiration criterion: allow a tabu move if it improves the global best
            if is_tabu and nv <= best_v:
                continue

            if nv > best_neighbour_v:
                best_neighbour = neighbour
                best_neighbour_v = nv
                best_neighbour_w = nw
                best_move = i

        # If no non-tabu neighbour was found (can happen with large tenures), skip
        if best_neighbour is None:
            continue

        # Step 4: move to best non-tabu neighbour
        current = best_neighbour
        current_v = best_neighbour_v
        current_w = best_neighbour_w

        # Step 5: update tabu memory for the chosen move
        tabu_list[best_move] = iteration + tabu_tenure

        # Step 6: update global best
        if current_v > best_v:
            best = current[:]
            best_v = current_v
            best_w = current_w

    return {"value": best_v, "weight": best_w, "solution": best}

### Running Experiments

We test all combinations of `max_iterations` and `tabu_tenure` across both problem instances.
Each setting is run `NUM_RUNS` times to account for randomness; we report the best and average value found.

In [8]:
def run_experiments(num_runs: int = NUM_RUNS):
    """
    Runs Tabu Search for each combination of (instance, max_iterations, tabu_tenure).
    Saves aggregated results to a markdown file.
    """
    with open(OUTPUT_FILE, "w") as f:
        f.write("# Tabu Search – Knapsack Results\n\n")
        f.write(
            f"| {'Instance':<25} | {'Runs':<5} | {'Max Iter':<9} | {'Tenure':<7} "
            f"| {'Best Value':<11} | {'Avg Value':<10} | {'Avg Time (s)':<13} |\n"
        )
        f.write("|---|---|---|---|---|---|---|\n")

    for file_info in FILES_TO_TEST:
        filename = file_info["name"]
        items, capacity = load_data(filename)

        if not items:
            with open(OUTPUT_FILE, "a") as f:
                f.write(f"| {filename} | Error: file not found |\n")
            continue

        for max_iter in MAX_ITERATIONS:
            for tenure in TABU_TENURES:
                best_val = -1
                sum_vals = 0
                total_time = 0

                for _ in range(num_runs):
                    start = time.time()
                    result = tabu_search(items, capacity, max_iter, tenure)
                    elapsed = time.time() - start

                    total_time += elapsed
                    sum_vals += result["value"]
                    if result["value"] > best_val:
                        best_val = result["value"]

                avg_val = sum_vals / num_runs
                avg_time = total_time / num_runs

                print(
                    f"{filename} | iter={max_iter} tenure={tenure} "
                    f"| best={best_val} avg={avg_val:.1f} time={avg_time:.4f}s"
                )

                with open(OUTPUT_FILE, "a") as f:
                    f.write(
                        f"| {filename:<25} | {num_runs:<5} | {max_iter:<9} | {tenure:<7} "
                        f"| {best_val:<11} | {avg_val:<10.2f} | {avg_time:<13.4f} |\n"
                    )


if __name__ == "__main__":
    run_experiments(NUM_RUNS)

../a1/knapsack-20.txt | iter=100 tenure=5 | best=787 avg=753.7 time=0.0031s
../a1/knapsack-20.txt | iter=100 tenure=10 | best=787 avg=783.4 time=0.0030s
../a1/knapsack-20.txt | iter=100 tenure=20 | best=787 avg=774.5 time=0.0030s
../a1/knapsack-20.txt | iter=500 tenure=5 | best=787 avg=751.4 time=0.0154s
../a1/knapsack-20.txt | iter=500 tenure=10 | best=787 avg=782.9 time=0.0150s
../a1/knapsack-20.txt | iter=500 tenure=20 | best=787 avg=775.5 time=0.0148s
../a1/knapsack-20.txt | iter=1000 tenure=5 | best=787 avg=779.6 time=0.0312s
../a1/knapsack-20.txt | iter=1000 tenure=10 | best=787 avg=784.6 time=0.0306s
../a1/knapsack-20.txt | iter=1000 tenure=20 | best=787 avg=769.8 time=0.0298s
../a1/knapsack-200.txt | iter=100 tenure=5 | best=97544 avg=9656.9 time=0.2140s
../a1/knapsack-200.txt | iter=100 tenure=10 | best=96746 avg=9634.5 time=0.2334s
../a1/knapsack-200.txt | iter=100 tenure=20 | best=97618 avg=9749.0 time=0.2390s
../a1/knapsack-200.txt | iter=500 tenure=5 | best=96572 avg=19245

---
## TSP – Read data and greedy solution

### Read TSP data

In [5]:
import math

def load_tsp(file_name: str) -> list[tuple[float, float]]:
    """
    Parses a TSP file in TSPLIB EUC_2D format.
    Returns a list of (x, y) city coordinates.
    """
    coords = []
    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return coords

    in_node_section = False
    with open(file_name) as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                in_node_section = True
                continue
            if line in ("EOF", ""):
                in_node_section = False
                continue
            if in_node_section:
                parts = line.split()
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    return coords


def euclidean_distance(c1: tuple[float, float], c2: tuple[float, float]) -> float:
    """Returns the Euclidean distance between two cities."""
    return math.sqrt((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2)


def build_distance_matrix(coords: list[tuple[float, float]]) -> list[list[float]]:
    """Pre-computes all pairwise distances for fast tour evaluation."""
    n = len(coords)
    dist = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = euclidean_distance(coords[i], coords[j])
            dist[i][j] = d
            dist[j][i] = d
    return dist


def tour_cost(tour: list[int], dist: list[list[float]]) -> float:
    """Calculates the total cost of a TSP tour (returns to start city)."""
    total = 0.0
    n = len(tour)
    for i in range(n):
        total += dist[tour[i]][tour[(i + 1) % n]]
    return total

### Greedy solution for TSP

The **nearest-neighbour greedy heuristic** builds a tour by always moving to the closest unvisited city:
1. Start from city 0.
2. Repeatedly visit the nearest unvisited city.
3. Return to the start city to close the tour.

In [10]:
def greedy_tsp(dist: list[list[float]], start: int = 0) -> list[int]:
    """
    Nearest-neighbour greedy heuristic for TSP.
    Returns the tour as a list of city indices.
    """
    n = len(dist)
    visited = [False] * n
    tour = [start]
    visited[start] = True

    for _ in range(n - 1):
        current = tour[-1]
        # Find the closest unvisited city
        nearest = min(
            (j for j in range(n) if not visited[j]),
            key=lambda j: dist[current][j]
        )
        tour.append(nearest)
        visited[nearest] = True

    return tour


def verify_tour(tour: list[int], n: int) -> bool:
    """Checks that a tour is a valid permutation of all cities."""
    return sorted(tour) == list(range(n))

### Running the TSP greedy demo

In [11]:
TSP_FILE = "../tsp_instances/A/berlin52.tsp"   

coords = load_tsp(TSP_FILE)

if coords:
    n_cities = len(coords)
    print(f"Loaded {n_cities} cities from '{TSP_FILE}'")

    dist_matrix = build_distance_matrix(coords)

    # Generate greedy tour and measure its cost
    greedy_tour = greedy_tsp(dist_matrix, start=0)
    cost = tour_cost(greedy_tour, dist_matrix)

    # Verify the tour visits every city exactly once
    is_valid = verify_tour(greedy_tour, n_cities)

    print(f"Greedy tour cost : {cost:.2f}")
    print(f"Tour is valid    : {is_valid}")
    print(f"Tour             : {greedy_tour}")
else:
    print("No data loaded – check TSP_FILE path.")

Loaded 52 cities from '../tsp_instances/A/berlin52.tsp'
Greedy tour cost : 8980.92
Tour is valid    : True
Tour             : [0, 21, 48, 31, 35, 34, 33, 38, 39, 37, 36, 47, 23, 4, 14, 5, 3, 24, 45, 43, 15, 49, 19, 22, 30, 17, 2, 18, 44, 40, 7, 9, 8, 42, 32, 50, 11, 27, 26, 25, 46, 12, 13, 51, 10, 28, 29, 20, 16, 41, 6, 1]


---
## Tabu Search for TSP

### Helper functions for 2-opt moves

The **2-opt neighborhood** for TSP reverses a segment of the tour.
- A move is represented as a pair (i, j) where 0 ≤ i < j < n.
- The move reverses the segment between positions i and j in the tour.
- This swap removes edges (tour[i], tour[i+1]) and (tour[j], tour[(j+1)%n])
  and adds edges (tour[i], tour[j]) and (tour[i+1], tour[(j+1)%n]).


In [1]:
import os
import math
import time 

In [2]:
### Helpers for algorithm
def apply_2opt_move(tour: list[int], i: int, j: int) -> list[int]:
    """
    Applies a 2-opt move: reverses the segment between positions i and j.
    Returns a new tour without modifying the original.
    """
    # Reverse the segment from i+1 to j (inclusive)
    return tour[:i+1] + tour[i+1:j+1][::-1] + tour[j+1:]


def get_2opt_neighbors(tour: list[int]) -> list[tuple[list[int], tuple[int, int]]]:
    """
    Generates all 2-opt neighbors of a tour.
    Returns a list of (new_tour, move) pairs where move is (i, j).
    """
    neighbors = []
    n = len(tour)
    for i in range(n - 2):
        for j in range(i + 2, n):
            new_tour = apply_2opt_move(tour, i, j)
            neighbors.append((new_tour, (i, j)))
    return neighbors

def two_opt_delta(tour, dist, i, j):
    """
    Cost change from reversing tour[i+1 : j+1].
    Removes edges (tour[i]→tour[i+1]) and (tour[j]→tour[j+1 % n]).
    Adds    edges (tour[i]→tour[j])   and (tour[i+1]→tour[j+1 % n]).
    """
    n = len(tour)
    a, b = tour[i],       tour[i + 1]
    c, d = tour[j],       tour[(j + 1) % n]
    return dist[a][c] + dist[b][d] - dist[a][b] - dist[c][d]


def encode_move(move: tuple[int, int]) -> int:
    """
    Encode a 2-opt move (i, j) as a single integer for tabu list.
    We use a simple formula: i * 10000 + j (assuming n < 10000).
    """
    return move[0] * 10000 + move[1]


def decode_move(encoded: int) -> tuple[int, int]:
    """Decode an encoded move back to (i, j)."""
    return (encoded // 10000, encoded % 10000)

### Helpers for TSP

def load_tsp(file_name: str) -> list[tuple[float, float]]:
    """
    Parses a TSP file in TSPLIB EUC_2D format.
    Returns a list of (x, y) city coordinates.
    """
    coords = []
    if not os.path.exists(file_name):
        print(f"Error: File '{file_name}' not found.")
        return coords

    in_node_section = False
    with open(file_name) as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                in_node_section = True
                continue
            if line in ("EOF", ""):
                in_node_section = False
                continue
            if in_node_section:
                parts = line.split()
                if len(parts) >= 3:
                    coords.append((float(parts[1]), float(parts[2])))
    return coords
def euclidean_distance(c1: tuple[float, float], c2: tuple[float, float]) -> float:
    """Returns the Euclidean distance between two cities."""
    return math.sqrt((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2)


def build_distance_matrix(coords: list[tuple[float, float]]) -> list[list[float]]:
    """Pre-computes all pairwise distances for fast tour evaluation."""
    n = len(coords)
    dist = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            d = euclidean_distance(coords[i], coords[j])
            dist[i][j] = d
            dist[j][i] = d
    return dist


def tour_cost(tour: list[int], dist: list[list[float]]) -> float:
    """Calculates the total cost of a TSP tour (returns to start city)."""
    total = 0.0
    n = len(tour)
    for i in range(n):
        total += dist[tour[i]][tour[(i + 1) % n]]
    return total


### Greedy tsp

def greedy_tsp(dist: list[list[float]], start: int = 0) -> list[int]:
    """
    Nearest-neighbour greedy heuristic for TSP.
    Returns the tour as a list of city indices.
    """
    n = len(dist)
    visited = [False] * n
    tour = [start]
    visited[start] = True

    for _ in range(n - 1):
        current = tour[-1]
        # Find the closest unvisited city
        nearest = min(
            (j for j in range(n) if not visited[j]),
            key=lambda j: dist[current][j]
        )
        tour.append(nearest)
        visited[nearest] = True

    return tour


def verify_tour(tour: list[int], n: int) -> bool:
    """Checks that a tour is a valid permutation of all cities."""
    return sorted(tour) == list(range(n))

### Core Tabu Search Algorithm for TSP

**Pseudocode:**
```
c = greedy_solution()
best = c
M = initMemory()                           # tabu list: stores forbidden moves
iter = 0
while iter < max_iterations:
    neighbors = getAllNeighbours(c)        # all 2-opt neighbors
    x = getBestNeighbourNonTabu(neighbors, M, iter)  # best non-tabu neighbor
    
    if x is None:
        break  # no non-tabu neighbor found
    
    move = getMove(c, x)
    updateMemory(M, move, iter, tabu_tenure)  # mark move as tabu
    c = x
    
    if cost(c) < cost(best):
        best = c
    iter += 1
return best
```

**Key features:**
- Start from greedy solution (better initial solution)
- Use 2-opt moves (reverse segment in tour)
- Aspiration criterion allows tabu moves if they improve global best


In [3]:
def tabu_search_tsp(
    dist: list[list[float]],
    max_iterations: int,
    tabu_tenure: int,
    initial_tour: list[int] = None
) -> dict:
    n = len(dist)
    
    if initial_tour is None:
        current = greedy_tsp(dist, start=0)
    else:
        current = initial_tour[:]
    
    current_cost = tour_cost(current, dist)
    best = current[:]
    best_cost = current_cost
    tabu_list = {}
    
    for iteration in range(max_iterations):
        best_delta = float('inf')
        best_i, best_j = -1, -1

        for i in range(n - 2):
            for j in range(i + 2, n):
                delta = two_opt_delta(current, dist, i, j)
                encoded = encode_move((i, j))
                is_tabu = encoded in tabu_list and tabu_list[encoded] > iteration

                # Aspiration criterion: allow tabu move if it beats global best
                if is_tabu and (current_cost + delta) >= best_cost:
                    continue

                if delta < best_delta:
                    best_delta = delta
                    best_i, best_j = i, j

        if best_i == -1:
            break

        # Apply the chosen move
        current = apply_2opt_move(current, best_i, best_j)
        current_cost += best_delta
        tabu_list[encode_move((best_i, best_j))] = iteration + tabu_tenure

        if current_cost < best_cost:
            best = current[:]
            best_cost = current_cost

    return {
        'tour': best,
        'cost': best_cost,
        'iterations': iteration + 1
    }

### Experiments on TSP Instances

We select one instance from group A (list A1-A16) and one from group B (list B1-B29), then run Tabu Search with different parameter settings.


In [4]:
# TSP instances to test (one from A, one from B)
TSP_INSTANCES = [
    {"name": "../tsp_instances/A/pr107.tsp", "group": "A", "label": "pr107"},
    {"name": "../tsp_instances/B/zi929.tsp", "group": "B", "label": "zi929"}
]

# Different parameter settings to test
TSP_TABU_TENURES = [5, 10, 20, 50]
TSP_MAX_ITERATIONS = [100, 500, 1000]

# Number of runs per configuration
TSP_NUM_RUNS = 20

# Output file for TSP results
TSP_OUTPUT_FILE = "tsp_tabu_results.md"


In [ ]:
def run_tsp_experiments(num_runs: int = TSP_NUM_RUNS):
    """
    Runs Tabu Search for TSP on selected instances with different parameter settings.
    Saves aggregated results to a markdown file.
    """
    with open(TSP_OUTPUT_FILE, "w") as f:
        f.write("# Tabu Search for TSP – Results\n\n")
        f.write("## Configuration\n")
        f.write(f"- **Instances:** 1 from group A (A1-A16), 1 from group B (B1-B29)\n")
        f.write(f"- **Number of runs per configuration:** {num_runs}\n")
        f.write(f"- **Tabu tenures tested:** {TSP_TABU_TENURES}\n")
        f.write(f"- **Max iterations tested:** {TSP_MAX_ITERATIONS}\n\n")
        
        f.write("## Results\n\n")
        f.write(
            f"| {'Instance':<20} | {'Size':<6} | {'Max Iter':<10} | {'Tenure':<8} "
            f"| {'Best Cost':<12} | {'Avg Cost':<12} | {'Avg Time (s)':<13} |\n"
        )
        f.write("|---|---|---|---|---|---|---|\n")

    for instance_info in TSP_INSTANCES:
        filename = instance_info["name"]
        label = instance_info["label"]
        
        coords = load_tsp(filename)
        if not coords:
            with open(TSP_OUTPUT_FILE, "a") as f:
                f.write(f"| {label:<20} | ERROR |\n")
            print(f"Error loading {filename}")
            continue
        
        n_cities = len(coords)
        dist_matrix = build_distance_matrix(coords)
        
        print(f"\n{'='*60}")
        print(f"Instance: {label} (n={n_cities})")
        print(f"{'='*60}")
        
        for max_iter in TSP_MAX_ITERATIONS:
            for tenure in TSP_TABU_TENURES:
                best_cost = float('inf')
                sum_costs = 0
                total_time = 0
                
                for run in range(num_runs):
                    start = time.time()
                    result = tabu_search_tsp(dist_matrix, max_iter, tenure)
                    elapsed = time.time() - start
                    
                    total_time += elapsed
                    sum_costs += result['cost']
                    if result['cost'] < best_cost:
                        best_cost = result['cost']
                
                avg_cost = sum_costs / num_runs
                avg_time = total_time / num_runs
                
                print(
                    f"  iter={max_iter:4d} tenure={tenure:2d} | best={best_cost:10.2f} "
                    f"avg={avg_cost:10.2f} time={avg_time:.4f}s"
                )
                
                with open(TSP_OUTPUT_FILE, "a") as f:
                    f.write(
                        f"| {label:<20} | {n_cities:<6} | {max_iter:<10} | {tenure:<8} "
                        f"| {best_cost:<12.2f} | {avg_cost:<12.2f} | {avg_time:<13.4f} |\n"
                    )
        
        # Add greedy baseline for comparison
        greedy_solution = greedy_tsp(dist_matrix, start=0)
        greedy_cost = tour_cost(greedy_solution, dist_matrix)
        
        print(f"  GREEDY (baseline)   | best={greedy_cost:10.2f}")
        
        with open(TSP_OUTPUT_FILE, "a") as f:
            f.write(
                f"| {label} (greedy):<20 | {n_cities:<6} | {'N/A':<10} | {'N/A':<8} "
                f"| {greedy_cost:<12.2f} | {greedy_cost:<12.2f} | {'0.0000':<13} |\n"
            )
    
    print(f"\nResults saved to {TSP_OUTPUT_FILE}")


# Run the experiments
if __name__ == "__main__":
    run_tsp_experiments(TSP_NUM_RUNS)



Instance: pr107 (n=107)
  iter= 100 tenure= 5 | best=  44646.00 avg=  44646.00 time=0.1627s
  iter= 100 tenure=10 | best=  44646.00 avg=  44646.00 time=0.1613s
  iter= 100 tenure=20 | best=  44622.84 avg=  44622.84 time=0.1601s
  iter= 100 tenure=50 | best=  44436.24 avg=  44436.24 time=0.1644s
  iter= 500 tenure= 5 | best=  44646.00 avg=  44646.00 time=0.7936s
  iter= 500 tenure=10 | best=  44646.00 avg=  44646.00 time=0.8039s
  iter= 500 tenure=20 | best=  44496.76 avg=  44496.76 time=0.8083s
  iter= 500 tenure=50 | best=  44436.24 avg=  44436.24 time=0.8112s
  iter=1000 tenure= 5 | best=  44646.00 avg=  44646.00 time=1.5858s
  iter=1000 tenure=10 | best=  44646.00 avg=  44646.00 time=1.6263s
  iter=1000 tenure=20 | best=  44496.76 avg=  44496.76 time=1.6246s
  iter=1000 tenure=50 | best=  44436.24 avg=  44436.24 time=1.6481s
  GREEDY (baseline)   | best=  46678.15

Instance: zi929 (n=929)
  iter= 100 tenure= 5 | best= 100188.67 avg= 100188.67 time=17.5890s
  iter= 100 tenure=10 | b